# Prep

In [ ]:
!pip install wordfreq pandas matplotlib
!pip install -U nltk
!python -m nltk.downloader popular

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('maxent_ne_chunker')
nltk.download('maxent_ne_chunker_tab')
nltk.download('words')

# Helpers

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap
from matplotlib.colors import Normalize
import seaborn as sns
from nltk.tokenize import word_tokenize
from matplotlib.lines import Line2D
import random


In [ ]:
plt.rcParams.update({
    'axes.linewidth': 1.2,
    'font.size': 15,
    'axes.labelsize': 15,
    'xtick.labelsize': 15,
    'ytick.labelsize': 15,
    'legend.fontsize': 15,
})

In [ ]:
DECILES = 10
TOP_KS = [50, 100, 500, 1000]
INCLUDE_BOS_EOS = True
BOS_TOKEN, EOS_TOKEN = "<s>", "</s>"

def ngram_cumulative(token_lists, n=1):
    counter = Counter()
    for seq in token_lists:
        toks = list(seq)
        if INCLUDE_BOS_EOS:
            toks = [BOS_TOKEN]*(n-1) + toks + [EOS_TOKEN]
        for i in range(len(toks)-n+1):
            counter[tuple(toks[i:i+n])] += 1
    if not counter:
        return np.array([1.0])
    counts = np.array(sorted(counter.values(), reverse=True), dtype=float)
    return np.cumsum(counts) / counts.sum(), counter

In [ ]:
def culmulative_graph(cum_uni, cum_bi, cum_tri, save):
    fig, axes = plt.subplots(1, 3, figsize=(21, 6), sharey=True)
    for d in range(DECILES):
        color = cmap(norm(d))
        axes[0].plot(np.arange(1, len(cum_uni[d])+1), cum_uni[d], color=color, linewidth=2)
        axes[1].plot(np.arange(1, len(cum_bi[d])+1),  cum_bi[d],  color=color, linewidth=2)
        axes[2].plot(np.arange(1, len(cum_tri[d])+1), cum_tri[d], color=color, linewidth=2)

    for ax, title in zip(axes, ["Uni-gram", "Bi-gram", "Tri-gram"],):
        ax.set_title(title)
        ax.set_xlabel("Rank")
        ax.grid(True, alpha=0.3)

    axes[0].set_ylabel("Cumulative probability")

    labels = [f"{d}" if d not in [0, 9] else ("0 (Low)" if d==0 else "9 (High)") 
            for d in range(DECILES)]
    handles = [Line2D([0], [0], color=cmap(norm(d)), lw=4, label=labels[d]) 
            for d in range(DECILES)]

    if "revmc" in mode:
        mode_str = "KLR"
    elif "mc" in mode:
        mode_str = "KL"
    elif "c" in mode:
        mode_str = "C"
    elif "w" in mode:
        mode_str = "W"
    else:
        raise ValueError("Invalid mode")

    leg=fig.legend(handles=handles, title="$D_\mathrm{" + mode_str + "}$ decile",
            loc="center left", bbox_to_anchor=(1.0, 0.5),
            frameon=False, handlelength=2.8)

    plt.tight_layout()
    if save is not None:
        plt.savefig(save, bbox_inches='tight', bbox_extra_artists=[leg])
    plt.show()

In [ ]:
def heatmap(cum_uni, cum_bi, cum_tri, save):
    def coverage_at_K(cum, K):
        if not len(cum): return 0
        K = min(K, len(cum))
        return float(cum[K-1])

    rows = []
    for d in range(DECILES):
        for K in TOP_KS:
            rows.append({
                "decile": d,
                "K": K,
                "Uni": coverage_at_K(cum_uni[d], K),
                "Bi":  coverage_at_K(cum_bi[d], K),
                "Tri": coverage_at_K(cum_tri[d], K),
            })
    cover_df = pd.DataFrame(rows)

    fig, axes = plt.subplots(1, 3, figsize=(16,5), sharey=True)

    sns.heatmap(
        cover_df.pivot(index="decile", columns="K", values="Uni"),
        cmap="viridis", annot=True, fmt=".2f", ax=axes[0]
    )
    axes[0].set_title("Uni-gram")

    sns.heatmap(
        cover_df.pivot(index="decile", columns="K", values="Bi"),
        cmap="viridis", annot=True, fmt=".2f", ax=axes[1]
    )
    axes[1].set_title("Bi-gram")

    sns.heatmap(
        cover_df.pivot(index="decile", columns="K", values="Tri"),
        cmap="viridis", annot=True, fmt=".2f", ax=axes[2]
    )
    axes[2].set_title("Tri-gram")

    for ax in axes:
        ax.set_xlabel("Top-K N-grams")
        ax.set_ylabel("IG decile")

    plt.tight_layout()
    if save is not None:
        plt.savefig(save)
    plt.show()

# Img

In [ ]:
import numpy as np
data = np.load("path-to-output/img_whiten.npz")
captions = np.load("path-to-output/img_whiten_caption.npz")

In [ ]:
igs = data['ig']
ids = data['id']

## NLTK

In [ ]:
records = []
for i, sample_id in enumerate(ids):
    caps = captions[str(sample_id)]
    feats = {
        "cap": caps
    }
    feats["id"] = sample_id
    feats["ig"] = float(igs[i])     # ig[i] は float と明示
    records.append(feats)

df = pd.DataFrame(records)
df.head()

In [ ]:
df_sorted = df.sort_values('ig')
df_sorted['cap_all'] = df_sorted['cap'].apply(lambda x: ' '.join(x))
df_sorted.head()

In [ ]:
df_sorted['cap_all_token'] = df_sorted['cap_all'].apply(lambda x: word_tokenize(x.lower()))
df_sorted['cap_all_type'] = df_sorted['cap_all_token'].apply(lambda x: set(x))
df_sorted['cap_all_length'] = df_sorted['cap_all_token'].apply(lambda x: len(x))
df_sorted['cap_all_vocab_length'] = df_sorted['cap_all_type'].apply(lambda x: len(x))
df_sorted['ttr'] = df_sorted['cap_all_vocab_length'] / df_sorted['cap_all_length']
df_sorted.head()


In [ ]:
df_sorted[["ig", "cap_all_length", "cap_all_vocab_length", "ttr"]].corr()

In [ ]:
df_sorted['cap_token'] = df_sorted['cap'].apply(lambda x: [word_tokenize(s.lower()) for s in x])
df_sorted.head()


In [ ]:
df_sorted["ig_decile"] = pd.qcut(df_sorted["ig"], DECILES, labels=False)


cum_uni, cum_bi, cum_tri = {}, {}, {}
cnt_uni, cnt_bi, cnt_tri = {}, {}, {}
for d in range(DECILES):
    token_lists = df_sorted.loc[df_sorted["ig_decile"] == d, "cap_token"].tolist()
    token_lists = sum(token_lists, [])
    cum_uni[d], cnt_uni[d] = ngram_cumulative(token_lists, 1)
    cum_bi[d], cnt_bi[d]  = ngram_cumulative(token_lists, 2)
    cum_tri[d], cnt_tri[d] = ngram_cumulative(token_lists, 3)

cmap = get_cmap("viridis")
norm = Normalize(vmin=0, vmax=DECILES-1)


In [ ]:
cnt_tri[0].most_common(20)

In [ ]:
cnt_tri[9].most_common(20)

In [ ]:
ones = [item for item, count in cnt_tri[0].items() if count == 1]
random.sample(ones, k=20)

In [ ]:
ones = [item for item, count in cnt_tri[9].items() if count == 1]
random.sample(ones, k=20)

In [ ]:
culmulative_graph(cum_uni, cum_bi, cum_tri, save=f"/groups/gag51404/fumiyau/repos/clip_ig/notebooks/out/mscoco_img_{mode}.pdf")

In [ ]:
heatmap(cum_uni, cum_bi, cum_tri, save=f"/groups/gag51404/fumiyau/repos/clip_ig/notebooks/out/mscoco_img_heatmap_{mode}.pdf")

# txt

In [ ]:
data = np.load("path-to-output/txt_whiten.npz")

data.files

In [ ]:
igs = data["ig"]
captions = data['caption']

In [ ]:
records = []
for i, sample_id in enumerate(ids):
    caps = captions[i]
    feats = {
        "cap": caps
    }
    feats["ig"] = float(igs[i])
    records.append(feats)

df = pd.DataFrame(records)
df.head()

In [ ]:
df_sorted = df.sort_values('ig')

## NLTK

In [ ]:
df_sorted['cap_token'] = df_sorted['cap'].apply(lambda x: word_tokenize(x.lower()))
df_sorted['cap_type'] = df_sorted['cap_token'].apply(lambda x: set(x))
df_sorted['cap_length'] = df_sorted['cap_token'].apply(lambda x: len(x))
df_sorted['cap_vocab_length'] = df_sorted['cap_type'].apply(lambda x: len(x))
df_sorted['ttr'] = df_sorted['cap_vocab_length'] / df_sorted['cap_length']
df_sorted.head()

In [ ]:
df_sorted[["ig", "cap_length", "cap_vocab_length", "ttr"]].corr()

In [ ]:
df_sorted["ig_decile"] = pd.qcut(df_sorted["ig"], DECILES, labels=False)


cum_uni, cum_bi, cum_tri = {}, {}, {}
cnt_uni, cnt_bi, cnt_tri = {}, {}, {}
for d in range(DECILES):
    token_lists = df_sorted.loc[df_sorted["ig_decile"] == d, "cap_token"].tolist()
    cum_uni[d], cnt_uni[d] = ngram_cumulative(token_lists, 1)
    cum_bi[d], cnt_bi[d]  = ngram_cumulative(token_lists, 2)
    cum_tri[d], cnt_tri[d] = ngram_cumulative(token_lists, 3)

cmap = get_cmap("viridis")
norm = Normalize(vmin=0, vmax=DECILES-1)

In [ ]:
cnt_tri[0].most_common(20)

In [ ]:
cnt_tri[9].most_common(20)

In [ ]:
ones = [item for item, count in cnt_tri[0].items() if count == 1]
random.sample(ones, k=20)

In [ ]:
ones = [item for item, count in cnt_tri[9].items() if count == 1]
random.sample(ones, k=20)

In [ ]:
culmulative_graph(cum_uni, cum_bi, cum_tri, save=f"/groups/gag51404/fumiyau/repos/clip_ig/notebooks/out/mscoco_txt_{mode}.pdf")

In [ ]:
heatmap(cum_uni, cum_bi, cum_tri, save=f"/groups/gag51404/fumiyau/repos/clip_ig/notebooks/out/mscoco_txt_heatmap_{mode}.pdf")